# Часть 5. Статистика по туберкулезу в России

**Сводка по таблицам (для построения дашборда)**

**Источник:**  
«Социально значимые заболевания населения России в ____ году» (Статистические материалы)

| Таблица | Что содержит | Период / География | Ключевые поля | Применение в дашборде |
|---------|--------------|--------------------|----------------|------------------------|
| **Таблица 1.1. Туберкулез в Российской Федерации** | Динамика впервые выявленных (всего и органов дыхания), контингентов, смертности (абс. числа + на 100 тыс.) | 1993–2010, РФ в целом | Год, абсолютные числа, показатели на 100 тыс. | Тренды, прогнозы, сравнение динамики трёх основных индикаторов |
| **Таблица 1.2. Заболеваемость и контингенты пациентов активным туберкулезом по субъектам Российской Федерации** | Впервые выявленные и контингенты (абс. + на 100 тыс.) в разрезе регионов | 2015–2016, все субъекты РФ | Регион, год, абсолютные числа, показатели на 100 тыс. | Картограмма, ранжирование регионов, выявление лидеров/аутсайдеров |
| **Таблица 1.3. Распределение пациентов с впервые в жизни установленным диагнозом активного туберкулеза по формам и полу в Российской Федерации** | Впервые выявленные по формам (лёгочный, внелёгочный, ЦНС, кости, мочеполовые, лимфоузлы) и полу (абс. + на 100 тыс.) | 2015–2016, РФ в целом | Форма, пол, год, абсолютные числа, показатели на 100 тыс. | Гендерные срезы, структура заболеваемости по локализациям, «портрет» больного |

**Общие характеристики:**  
- Все таблицы содержат **абсолютные числа** и **показатели на 100 тыс. населения** (стандартизованные).  
- Данные можно использовать как для мониторинга динамики, так и для межрегиональных сравнений.  
- Для расчётов на 100 тыс. уже есть готовые значения, дополнительный пересчёт не требуется.  

**Готовность к дашборду:** ✅ данные чистые, структурированы, имеют временную и региональную привязку, позволяют строить как простые линейные графики, так и интерактивные карты.

## План построения интерактивной карты в DataLens

Для визуализации региональных данных по туберкулёзу будет использован тип диаграммы **«Карта с областями» (Choropleth map)**. Это позволит закрасить регионы в разные цвета в зависимости от значения показателя.

Подготовка геоданных

1. **Загрузим** CSV-файл с полигонами всех регионов РФ (файл `Regions.csv` доступен по [ссылке](https://storage.yandexcloud.net/doc-files/Regions.csv)).  
2. В DataLens создадим отдельный датасет с геоданными, указав тип поля `Полигон` как **Geopolygon**.

Подготовка статистических данных

3. Используем уже подготовленный датасет `tb_regions`, который содержит показатели заболеваемости и контингентов в разрезе регионов за 2016–2024 годы.  
4. Убедимся, что названия регионов в `tb_regions` совпадают с названиями в геофайле (при необходимости выполним нормализацию).

Построение карты в чарте

5. Выберем тип диаграммы **Map**, слой **Polygons (Geopolygons)**.  
6. Перетащим поле `polygon` из геоданных в секцию **Polygons** – отобразится карта России.  
7. Свяжем статистические данные с геометрией через **INNER JOIN** по полю `region`.  
8. Показатель (например, `value`) перетащим в секцию **Colors** – регионы окрасятся в соответствии со значением.  
9. В секцию **Tooltips** добавим название региона и значение показателя для всплывающих подсказок.

Добавление динамики (селектор года)

10. На дашборде разместим созданную карту и элемент управления **Selector** (селектор).  
11. В селекторе укажем поле `year`.  
12. Настроим связь (линковку) между селектором и картой: поле `year` карты равно полю `year` селектора.  
13. При выборе года карта автоматически обновится, показывая данные только за выбранный период.

Использование контекстных данных

14. Дополнительно включим в дашборд графики из датасетов `tb_russia` (динамика по РФ) и `tb_forms` (структура по формам заболевания).  
15. Они не будут привязаны к селектору года (или будут, если требуется) и дадут общую картину.

---

Таким образом, будут объединены все компоненты: статистика, геометрия, интерактивность и контекст, что позволит оперативно анализировать эпидемиологическую ситуацию в региональном разрезе.

### Загрузка, приведение типов, информация

In [1]:
import pandas as pd
import numpy as np
import base64
import chardet
from IPython.display import HTML, display

In [2]:
# Загружаем датасеты
soc_1_1 = pd.read_csv('soc_table_1_1_long_1993-2024.csv')
soc_1_2 = pd.read_csv('soc_1.2_2016-2024.csv')
soc_1_3 = pd.read_csv('soc_table_1_3_long_2015-2024.csv')

# Функция для приведения колонки 'Значение' к числовому типу
def fix_value_column(df, df_name):
    if 'Значение' in df.columns:
        # Заменяем запятую на точку
        df['Значение'] = df['Значение'].astype(str).str.replace(',', '.', regex=False)
        # Преобразуем в numeric
        df['Значение'] = pd.to_numeric(df['Значение'], errors='coerce')
        print(f"   ✓ {df_name}: колонка 'Значение' преобразована в numeric")
    else:
        print(f"   ⚠️ {df_name}: колонка 'Значение' не найдена")
    return df

print("="*80)
print("ПРИВЕДЕНИЕ ТИПОВ ДЛЯ КОЛОНКИ 'Значение'")
print("="*80)

print("\n📊 soc_1_1:")
soc_1_1 = fix_value_column(soc_1_1, 'soc_1_1')

print("\n📊 soc_1_2:")
soc_1_2 = fix_value_column(soc_1_2, 'soc_1_2')

print("\n📊 soc_1_3:")
soc_1_3 = fix_value_column(soc_1_3, 'soc_1_3')

print("\n" + "="*80)
print("ОБЩАЯ ИНФОРМАЦИЯ О ДАТАСЕТАХ")
print("="*80)

datasets = [
    ('soc_1_1', soc_1_1),
    ('soc_1_2', soc_1_2),
    ('soc_1_3', soc_1_3)
]

for name, df in datasets:
    print(f"\n{'='*80}")
    print(f"📊 {name}")
    print(f"{'='*80}")
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} колонок")
    print(f"Колонки: {list(df.columns)}")
    print(f"Типы данных:\n{df.dtypes}")
    
    # Выводим уникальные значения для нечисловых колонок
    non_numeric = df.select_dtypes(include=['object', 'category', 'string']).columns
    if len(non_numeric) > 0:
        print(f"\n🔍 Нечисловые колонки и их уникальные значения:")
        for col in non_numeric:
            unique_vals = df[col].dropna().unique()
            print(f"\n   📌 {col}:")
            print(f"      Уникальных значений: {len(unique_vals)}")
            if len(unique_vals) <= 20:
                print(f"      Значения: {list(unique_vals)}")
            else:
                print(f"      Первые 20: {list(unique_vals[:20])}")
                print(f"      ... и еще {len(unique_vals)-20} значений")
    else:
        print("\n✅ Нет нечисловых колонок")
    
    # Выводим диапазон годов
    if 'year' in df.columns:
        year_min = df['year'].min()
        year_max = df['year'].max()
        print(f"\n📅 Диапазон годов: {year_min} - {year_max}")
    
    print(f"\n📋 Первые 3 строки:")
    display(df.head(3))

print("\n" + "="*80)
print("✅ Вывод информации завершён")
print("="*80)

ПРИВЕДЕНИЕ ТИПОВ ДЛЯ КОЛОНКИ 'Значение'

📊 soc_1_1:
   ✓ soc_1_1: колонка 'Значение' преобразована в numeric

📊 soc_1_2:
   ✓ soc_1_2: колонка 'Значение' преобразована в numeric

📊 soc_1_3:
   ✓ soc_1_3: колонка 'Значение' преобразована в numeric

ОБЩАЯ ИНФОРМАЦИЯ О ДАТАСЕТАХ

📊 soc_1_1
Размер: 256 строк × 4 колонок
Колонки: ['Год', 'Уточнение', 'Показатель', 'Значение']
Типы данных:
Год             int64
Уточнение      object
Показатель     object
Значение      float64
dtype: object

🔍 Нечисловые колонки и их уникальные значения:

   📌 Уточнение:
      Уникальных значений: 2
      Значения: ['Абс. числа', 'На 100 000 населения']

   📌 Показатель:
      Уникальных значений: 4
      Значения: ['Число пациентов с впервые в жизни установленным диагнозом - всего', 'Число пациентов с впервые в жизни установленным диагнозом - в том числе органов дыхания', 'Контингенты  больных активным туберкулезом состоящих под диспансерным наблюдением', 'Смертность населения от туберкулеза']

📋 Первые 3 ст

,Год,Уточнение,Показатель,Значение
0,1993,Абс. числа,Число пациентов с впервые в жизни установленны...,63591.0
1,1994,Абс. числа,Число пациентов с впервые в жизни установленны...,70822.0
2,1995,Абс. числа,Число пациентов с впервые в жизни установленны...,84980.0



📊 soc_1_2
Размер: 3384 строк × 5 колонок
Колонки: ['Субъект РФ', 'Федеральный округ', 'Год', 'Уточнение', 'Значение']
Типы данных:
Субъект РФ            object
Федеральный округ     object
Год                    int64
Уточнение             object
Значение             float64
dtype: object

🔍 Нечисловые колонки и их уникальные значения:

   📌 Субъект РФ:
      Уникальных значений: 95
      Первые 20: ['Российская Федерация', 'Центральный федеральный округ', 'Белгородская область', 'Брянская область', 'Владимирская область', 'Воронежская область', 'Ивановская область', 'Калужская область', 'Костромская область', 'Курская область', 'Липецкая область', 'Московская область', 'Орловская область', 'Рязанская область', 'Смоленская область', 'Тамбовская область', 'Тверская область', 'Тульская область', 'Ярославская область', 'город Москва']
      ... и еще 75 значений

   📌 Федеральный округ:
      Уникальных значений: 8
      Значения: ['Центральный федеральный округ', 'Северо-Западный федера

,Субъект РФ,Федеральный округ,Год,Уточнение,Значение
0,Российская Федерация,NaN,2016,"Впервые установленный диагноз, абс. число",78121.0
1,Центральный федеральный округ,NaN,2016,"Впервые установленный диагноз, абс. число",13375.0
2,Белгородская область,Центральный федеральный округ,2016,"Впервые установленный диагноз, абс. число",333.0



📊 soc_1_3
Размер: 480 строк × 6 колонок
Колонки: ['Показатель', 'Короткое название', 'Уточнение', 'Год', 'Пол', 'Значение']
Типы данных:
Показатель            object
Короткое название     object
Уточнение             object
Год                    int64
Пол                   object
Значение             float64
dtype: object

🔍 Нечисловые колонки и их уникальные значения:

   📌 Показатель:
      Уникальных значений: 8
      Значения: ['Число пациентов, с впервые в жизни установленным диагнозом активного туберкулеза - Все формы туберкулеза', 'Число пациентов, с впервые в жизни установленным диагнозом активного туберкулеза - Из них: туберкулез органов дыхания', 'Число пациентов, с впервые в жизни установленным диагнозом активного туберкулеза - Из них: внелегочный туберкулез', 'Число пациентов, с впервые в жизни установленным диагнозом активного туберкулеза - Из них: внелегочный туберкулез в том числе: туберкулез мозговых оболочек и ЦНС', 'Число пациентов, с впервые в жизни установленным д

,Показатель,Короткое название,Уточнение,Год,Пол,Значение
0,"Число пациентов, с впервые в жизни установленн...",все формы туберкулеза,Абс. числа,2015,мужчины,57669.0
1,"Число пациентов, с впервые в жизни установленн...",туберкулез органов дыхания,Абс. числа,2015,мужчины,56973.0
2,"Число пациентов, с впервые в жизни установленн...",внелегочный туберкулез,Абс. числа,2015,мужчины,1396.0



✅ Вывод информации завершён


### Унификация

In [3]:
# ======================================================================
# РАЗБИВКА КОЛОНКИ 'Уточнение' В ТАБЛИЦЕ soc_1_2
# ======================================================================

# Текущие значения в колонке 'Уточнение':
# - 'Впервые установленный диагноз, абс. число'
# - 'Впервые установленный диагноз, на 100000 населения'
# - 'Под диспансерным наблюдением на конец года, абс. число'
# - 'Под диспансерным наблюдением на конец года, на 100000 населения'

def split_measure(df):
    df = df.copy()
    
    # Создаём колонку 'Показатель' и 'Уточнение'
    df['Показатель'] = df['Уточнение'].apply(lambda x: 'Впервые установленный диагноз' if 'Впервые' in str(x) else 'Под диспансерным наблюдением на конец года')
    df['Уточнение'] = df['Уточнение'].apply(lambda x: 'абс. число' if 'абс. число' in str(x) else 'на 100000 населения')
    
    return df

soc_1_2 = split_measure(soc_1_2)

print("✅ Разбивка выполнена!")
print(f"\nНовые колонки: {list(soc_1_2.columns)}")
print("\n📋 Первые 5 строк:")
display(soc_1_2.head(5))

# Проверка уникальных значений новых колонок
print("\n🔍 Уникальные значения 'Показатель':")
print(soc_1_2['Показатель'].unique())
print("\n🔍 Уникальные значения 'Уточнение':")
print(soc_1_2['Уточнение'].unique())

✅ Разбивка выполнена!

Новые колонки: ['Субъект РФ', 'Федеральный округ', 'Год', 'Уточнение', 'Значение', 'Показатель']

📋 Первые 5 строк:


,Субъект РФ,Федеральный округ,Год,Уточнение,Значение,Показатель
0,Российская Федерация,NaN,2016,абс. число,78121.0,Впервые установленный диагноз
1,Центральный федеральный округ,NaN,2016,абс. число,13375.0,Впервые установленный диагноз
2,Белгородская область,Центральный федеральный округ,2016,абс. число,333.0,Впервые установленный диагноз
3,Брянская область,Центральный федеральный округ,2016,абс. число,655.0,Впервые установленный диагноз
4,Владимирская область,Центральный федеральный округ,2016,абс. число,510.0,Впервые установленный диагноз



🔍 Уникальные значения 'Показатель':
['Впервые установленный диагноз'
 'Под диспансерным наблюдением на конец года']

🔍 Уникальные значения 'Уточнение':
['абс. число' 'на 100000 населения']


In [4]:
# ======================================================================
# ПЕРЕИМЕНОВАНИЕ КОЛОНОК В ТАБЛИЦАХ soc_1_1, soc_1_2, soc_1_3
# ======================================================================

# Переименовываем soc_1_1
soc_1_1 = soc_1_1.rename(columns={
    'Год': 'year',
    'Уточнение': 'measure_type',
    'Показатель': 'name',
    'Значение': 'value'
})

# Переименовываем soc_1_2
soc_1_2 = soc_1_2.rename(columns={
    'Субъект РФ': 'region',
    'Федеральный округ': 'district',
    'Год': 'year',
    'Уточнение': 'measure_type',
    'Значение': 'value',
    'Показатель': 'name'
})

# Переименовываем soc_1_3
soc_1_3 = soc_1_3.rename(columns={
    'Показатель': 'full_name',
    'Короткое название': 'name',
    'Уточнение': 'measure_type',
    'Год': 'year',
    'Пол': 'gender',
    'Значение': 'value'
})

print("="*80)
print("ПЕРЕИМЕНОВАНИЕ КОЛОНОК В ТАБЛИЦАХ")
print("="*80)

print("\n✅ soc_1_1:")
print(f"   Колонки: {list(soc_1_1.columns)}")

print("\n✅ soc_1_2:")
print(f"   Колонки: {list(soc_1_2.columns)}")

print("\n✅ soc_1_3:")
print(f"   Колонки: {list(soc_1_3.columns)}")

print("\n" + "="*80)
print("ПЕРВЫЕ 3 СТРОКИ ПОСЛЕ ПЕРЕИМЕНОВАНИЯ")
print("="*80)

print("\n📊 soc_1_1:")
display(soc_1_1.head(3))

print("\n📊 soc_1_2:")
display(soc_1_2.head(3))

print("\n📊 soc_1_3:")
display(soc_1_3.head(3))

ПЕРЕИМЕНОВАНИЕ КОЛОНОК В ТАБЛИЦАХ

✅ soc_1_1:
   Колонки: ['year', 'measure_type', 'name', 'value']

✅ soc_1_2:
   Колонки: ['region', 'district', 'year', 'measure_type', 'value', 'name']

✅ soc_1_3:
   Колонки: ['full_name', 'name', 'measure_type', 'year', 'gender', 'value']

ПЕРВЫЕ 3 СТРОКИ ПОСЛЕ ПЕРЕИМЕНОВАНИЯ

📊 soc_1_1:


,year,measure_type,name,value
0,1993,Абс. числа,Число пациентов с впервые в жизни установленны...,63591.0
1,1994,Абс. числа,Число пациентов с впервые в жизни установленны...,70822.0
2,1995,Абс. числа,Число пациентов с впервые в жизни установленны...,84980.0



📊 soc_1_2:


,region,district,year,measure_type,value,name
0,Российская Федерация,NaN,2016,абс. число,78121.0,Впервые установленный диагноз
1,Центральный федеральный округ,NaN,2016,абс. число,13375.0,Впервые установленный диагноз
2,Белгородская область,Центральный федеральный округ,2016,абс. число,333.0,Впервые установленный диагноз



📊 soc_1_3:


,full_name,name,measure_type,year,gender,value
0,"Число пациентов, с впервые в жизни установленн...",все формы туберкулеза,Абс. числа,2015,мужчины,57669.0
1,"Число пациентов, с впервые в жизни установленн...",туберкулез органов дыхания,Абс. числа,2015,мужчины,56973.0
2,"Число пациентов, с впервые в жизни установленн...",внелегочный туберкулез,Абс. числа,2015,мужчины,1396.0


In [5]:
soc_1_3['measure_type'] = soc_1_3['measure_type'].replace({
    'Абс. числа': 'абс. число',
    'На 1000000 населения': 'на 100000 населения'
})

In [6]:
# ======================================================================
# ВЫВОД УНИКАЛЬНЫХ ЗНАЧЕНИЙ ДЛЯ НЕЧИСЛОВЫХ КОЛОНОК (soc_1_1, soc_1_2, soc_1_3)
# ======================================================================

datasets = [
    ('soc_1_1', soc_1_1),
    ('soc_1_2', soc_1_2),
    ('soc_1_3', soc_1_3)
]

print("="*80)
print("УНИКАЛЬНЫЕ ЗНАЧЕНИЯ НЕЧИСЛОВЫХ КОЛОНОК")
print("="*80)

for name, df in datasets:
    print(f"\n{'='*80}")
    print(f"📊 {name}")
    print(f"{'='*80}")
    
    non_numeric = df.select_dtypes(include=['object', 'category', 'string']).columns
    
    if len(non_numeric) == 0:
        print("✅ Нет нечисловых колонок")
        continue
    
    for col in non_numeric:
        unique_vals = df[col].dropna().unique()
        print(f"\n🔹 {col} ({len(unique_vals)} уникальных):")
        for i, val in enumerate(sorted(unique_vals), 1):
            # Если значение длинное, обрезаем
            val_str = str(val)
            
            print(f"   {i:3d}. {val_str}")

УНИКАЛЬНЫЕ ЗНАЧЕНИЯ НЕЧИСЛОВЫХ КОЛОНОК

📊 soc_1_1

🔹 measure_type (2 уникальных):
     1. Абс. числа
     2. На 100 000 населения

🔹 name (4 уникальных):
     1. Контингенты  больных активным туберкулезом состоящих под диспансерным наблюдением
     2. Смертность населения от туберкулеза
     3. Число пациентов с впервые в жизни установленным диагнозом - в том числе органов дыхания
     4. Число пациентов с впервые в жизни установленным диагнозом - всего

📊 soc_1_2

🔹 region (95 уникальных):
     1. Алтайский край
     2. Амурская область
     3. Архангельская область без автономного округа
     4. Астраханская область
     5. Белгородская область
     6. Брянская область
     7. Владимирская область
     8. Волгоградская область
     9. Вологодская область
    10. Воронежская область
    11. Дальневосточный федеральный округ
    12. Еврейская автономная область
    13. Забайкальский край
    14. Ивановская область
    15. Иркутская область
    16. Кабардино-Балкарская Республика
    17

In [7]:
# ======================================================================
# УНИФИКАЦИЯ ПОРЯДКА КОЛОНОК В ТАБЛИЦАХ soc_1_1, soc_1_2, soc_1_3
# ======================================================================

# Задаём порядок колонок для каждой таблицы
order_soc_1_1 = ['year', 'name', 'measure_type', 'value']
order_soc_1_2 = ['year', 'region', 'district', 'name', 'measure_type', 'value']
order_soc_1_3 = ['year', 'name', 'full_name', 'gender', 'measure_type', 'value']

# Применяем порядок
soc_1_1 = soc_1_1[order_soc_1_1]
soc_1_2 = soc_1_2[order_soc_1_2]
soc_1_3 = soc_1_3[order_soc_1_3]

print("="*80)
print("УНИФИЦИРОВАННЫЙ ПОРЯДОК КОЛОНОК")
print("="*80)

print("\n✅ soc_1_1:")
print(f"   Колонки: {list(soc_1_1.columns)}")

print("\n✅ soc_1_2:")
print(f"   Колонки: {list(soc_1_2.columns)}")

print("\n✅ soc_1_3:")
print(f"   Колонки: {list(soc_1_3.columns)}")

print("\n" + "="*80)
print("ПЕРВЫЕ 3 СТРОКИ ПОСЛЕ УНИФИКАЦИИ")
print("="*80)

print("\n📊 soc_1_1:")
display(soc_1_1.head(3))

print("\n📊 soc_1_2:")
display(soc_1_2.head(3))

print("\n📊 soc_1_3:")
display(soc_1_3.head(3))

УНИФИЦИРОВАННЫЙ ПОРЯДОК КОЛОНОК

✅ soc_1_1:
   Колонки: ['year', 'name', 'measure_type', 'value']

✅ soc_1_2:
   Колонки: ['year', 'region', 'district', 'name', 'measure_type', 'value']

✅ soc_1_3:
   Колонки: ['year', 'name', 'full_name', 'gender', 'measure_type', 'value']

ПЕРВЫЕ 3 СТРОКИ ПОСЛЕ УНИФИКАЦИИ

📊 soc_1_1:


,year,name,measure_type,value
0,1993,Число пациентов с впервые в жизни установленны...,Абс. числа,63591.0
1,1994,Число пациентов с впервые в жизни установленны...,Абс. числа,70822.0
2,1995,Число пациентов с впервые в жизни установленны...,Абс. числа,84980.0



📊 soc_1_2:


,year,region,district,name,measure_type,value
0,2016,Российская Федерация,NaN,Впервые установленный диагноз,абс. число,78121.0
1,2016,Центральный федеральный округ,NaN,Впервые установленный диагноз,абс. число,13375.0
2,2016,Белгородская область,Центральный федеральный округ,Впервые установленный диагноз,абс. число,333.0



📊 soc_1_3:


,year,name,full_name,gender,measure_type,value
0,2015,все формы туберкулеза,"Число пациентов, с впервые в жизни установленн...",мужчины,абс. число,57669.0
1,2015,туберкулез органов дыхания,"Число пациентов, с впервые в жизни установленн...",мужчины,абс. число,56973.0
2,2015,внелегочный туберкулез,"Число пациентов, с впервые в жизни установленн...",мужчины,абс. число,1396.0


In [8]:
# ======================================================================
# УНИФИКАЦИЯ ЗНАЧЕНИЙ В ТАБЛИЦАХ soc_1_1, soc_1_2, soc_1_3
# ======================================================================

# 1. soc_1_1: унификация measure_type
soc_1_1['measure_type'] = soc_1_1['measure_type'].replace({
    'Абс. числа': 'абс. число',
    'На 100 000 населения': 'на 100000 населения'
})

# 2. soc_1_2: унификация measure_type (уже в нужном формате)
#    унификация region: исправляем названия городов
soc_1_2['region'] = soc_1_2['region'].replace({
    'город Санкт - Петербург': 'город Санкт-Петербург',
    'город Санкт-Петербург': 'город Санкт-Петербург',
    'Ямало-Hенецкий АО': 'Ямало-Ненецкий АО'
})

# 3. soc_1_3: унификация measure_type
soc_1_3['measure_type'] = soc_1_3['measure_type'].replace({
    'Абс. числа': 'абс. число',
    'На 1000000 населения': 'на 1000000 населения'
})

# 4. soc_1_3: унификация названий форм (приведение к единому регистру и формату)
soc_1_3['name'] = soc_1_3['name'].str.lower().str.strip()

# Можно также создать короткие коды для форм (опционально)
form_mapping = {
    'все формы туберкулеза': 'Все формы',
    'туберкулез органов дыхания': 'Органы дыхания',
    'внелегочный туберкулез': 'Внелегочный',
    'туберкулез мозговых оболочек и цнс': 'ЦНС',
    'туберкулез костей и суставов': 'Кости и суставы',
    'туберкулез мочеполовых органов': 'Мочеполовые органы',
    'туберкулез периферических лимфоузлов': 'Лимфоузлы',
    'все формы туберкулеза у детей в возрасте 0-14 лет включительно': 'Дети 0-14'
}
# Применяем маппинг (если нужны короткие названия)
# soc_1_3['form_short'] = soc_1_3['form'].map(form_mapping)

print("="*80)
print("УНИФИКАЦИЯ ЗНАЧЕНИЙ В ТАБЛИЦАХ")
print("="*80)

print("\n✅ soc_1_1:")
print(f"   measure_type: {soc_1_1['measure_type'].unique()}")

print("\n✅ soc_1_2:")
print(f"   measure_type: {soc_1_2['measure_type'].unique()}")
print(f"   Уникальных регионов: {soc_1_2['region'].nunique()}")
print(f"   Исправленные регионы: 'город Санкт-Петербург', 'Ямало-Ненецкий АО'")

print("\n✅ soc_1_3:")
print(f"   measure_type: {soc_1_3['measure_type'].unique()}")
print(f"   form: {soc_1_3['name'].unique()}")

print("\n" + "="*80)
print("ПЕРВЫЕ 3 СТРОКИ ПОСЛЕ УНИФИКАЦИИ")
print("="*80)

print("\n📊 soc_1_1:")
display(soc_1_1.head(3))

print("\n📊 soc_1_2:")
display(soc_1_2.head(3))

print("\n📊 soc_1_3:")
display(soc_1_3.head(3))

УНИФИКАЦИЯ ЗНАЧЕНИЙ В ТАБЛИЦАХ

✅ soc_1_1:
   measure_type: ['абс. число' 'на 100000 населения']

✅ soc_1_2:
   measure_type: ['абс. число' 'на 100000 населения']
   Уникальных регионов: 94
   Исправленные регионы: 'город Санкт-Петербург', 'Ямало-Ненецкий АО'

✅ soc_1_3:
   measure_type: ['абс. число' 'на 100000 населения']
   form: ['все формы туберкулеза' 'туберкулез органов дыхания'
 'внелегочный туберкулез' 'туберкулез мозговых оболочек и цнс'
 'туберкулез костей и суставов' 'туберкулез мочеполовых органов'
 'туберкулез периферических лимфоузлов'
 'все формы туберкулеза у детей в возрасте 0-14 лет включительно']

ПЕРВЫЕ 3 СТРОКИ ПОСЛЕ УНИФИКАЦИИ

📊 soc_1_1:


,year,name,measure_type,value
0,1993,Число пациентов с впервые в жизни установленны...,абс. число,63591.0
1,1994,Число пациентов с впервые в жизни установленны...,абс. число,70822.0
2,1995,Число пациентов с впервые в жизни установленны...,абс. число,84980.0



📊 soc_1_2:


,year,region,district,name,measure_type,value
0,2016,Российская Федерация,NaN,Впервые установленный диагноз,абс. число,78121.0
1,2016,Центральный федеральный округ,NaN,Впервые установленный диагноз,абс. число,13375.0
2,2016,Белгородская область,Центральный федеральный округ,Впервые установленный диагноз,абс. число,333.0



📊 soc_1_3:


,year,name,full_name,gender,measure_type,value
0,2015,все формы туберкулеза,"Число пациентов, с впервые в жизни установленн...",мужчины,абс. число,57669.0
1,2015,туберкулез органов дыхания,"Число пациентов, с впервые в жизни установленн...",мужчины,абс. число,56973.0
2,2015,внелегочный туберкулез,"Число пациентов, с впервые в жизни установленн...",мужчины,абс. число,1396.0


### Переименуем названия

In [9]:
# Переименование переменных
tb_russia = soc_1_1
tb_regions = soc_1_2
tb_forms = soc_1_3

# Удаляем старые переменные
#del soc_1_1, soc_1_2, soc_1_3

print("✅ Переименовано:")
print("   soc_1_1 → tb_russia")
print("   soc_1_2 → tb_regions")
print("   soc_1_3 → tb_forms")

print("\n📊 Информация о новых переменных:")
print(f"   tb_russia_1: {tb_russia.shape[0]} строк, {tb_russia.shape[1]} колонок")
print(f"   tb_regions_2: {tb_regions.shape[0]} строк, {tb_regions.shape[1]} колонок")


✅ Переименовано:
   soc_1_1 → tb_russia
   soc_1_2 → tb_regions
   soc_1_3 → tb_forms

📊 Информация о новых переменных:
   tb_russia_1: 256 строк, 4 колонок
   tb_regions_2: 3384 строк, 6 колонок


### kids_adult столбец в tb_forms

In [10]:

# ============================================================
# 1. Маркируем существующие строки (дети и все возраста)
# ============================================================
df = tb_forms.copy()

conditions = [
    df['name'] == 'все формы туберкулеза у детей в возрасте 0-14 лет включительно',
    df['name'] == 'все формы туберкулеза'
]
choices = ['дети', 'все возраста']

df['kids_adult'] = np.select(conditions, choices, default='все возраста')

# ============================================================
# 2. Агрегируем данные для расчёта взрослых
# ============================================================
# Группировка по всем измерениям, кроме name и value
group_cols = ['year', 'gender', 'measure_type']

# Берём значения для "все формы"
all_forms = df[df['name'] == 'все формы туберкулеза'][group_cols + ['value']].rename(
    columns={'value': 'value_all'}
)

# Берём значения для "дети"
children_forms = df[df['name'] == 'все формы туберкулеза у детей в возрасте 0-14 лет включительно'][group_cols + ['value']].rename(
    columns={'value': 'value_children'}
)

# ============================================================
# 3. Вычисляем взрослых
# ============================================================
adults = pd.merge(all_forms, children_forms, on=group_cols, how='inner')
adults['value'] = adults['value_all'] - adults['value_children']

# ============================================================
# 4. Создаём строки для взрослых
# ============================================================
adults_rows = adults[group_cols + ['value']].copy()
adults_rows['name'] = 'взрослые (все формы - дети 0-14)'
adults_rows['kids_adult'] = 'взрослые'

# Добавляем остальные поля (заполняем None)
for col in ['full_name', 'women_percent', 'children_percent', 'val_prev']:
    if col in df.columns:
        adults_rows[col] = None

# ============================================================
# 5. Объединяем всё в один датасет
# ============================================================
tb_forms = pd.concat([df, adults_rows], ignore_index=True)

# ============================================================
# 6. Проверка результата
# ============================================================
print("=== Распределение по категориям kids_adult ===")
print(tb_forms['kids_adult'].value_counts())
print()

print("=== Пример строк для взрослых ===")
print(tb_forms[tb_forms['kids_adult'] == 'взрослые'][['year', 'gender', 'measure_type', 'name', 'value']].head(10))
print()

print("=== Пример строк для детей ===")
print(tb_forms[tb_forms['kids_adult'] == 'дети'][['year', 'gender', 'measure_type', 'name', 'value']].head(5))
print()

print("=== Пример строк для всех возрастов ===")
print(tb_forms[tb_forms['kids_adult'] == 'все возраста'][['year', 'gender', 'measure_type', 'name', 'value']].head(5))
print()

print(f"✅ Итоговый размер датасета: {len(tb_forms)} строк")
print(f"📊 Колонки: {list(tb_forms.columns)}")

=== Распределение по категориям kids_adult ===
kids_adult
все возраста    420
дети             60
взрослые         60
Name: count, dtype: int64

=== Пример строк для взрослых ===
     year   gender         measure_type                              name  \
480  2015  мужчины           абс. число  взрослые (все формы - дети 0-14)   
481  2015  мужчины  на 100000 населения  взрослые (все формы - дети 0-14)   
482  2016  мужчины           абс. число  взрослые (все формы - дети 0-14)   
483  2016  мужчины  на 100000 населения  взрослые (все формы - дети 0-14)   
484  2017  мужчины           абс. число  взрослые (все формы - дети 0-14)   
485  2017  мужчины  на 100000 населения  взрослые (все формы - дети 0-14)   
486  2018  мужчины           абс. число  взрослые (все формы - дети 0-14)   
487  2018  мужчины  на 100000 населения  взрослые (все формы - дети 0-14)   
488  2019  мужчины           абс. число  взрослые (все формы - дети 0-14)   
489  2019  мужчины  на 100000 населения  взрослые (

### Добавляем процент женщин и детей в tb_forms


In [11]:
# 1. Приводим measure_type к единому виду (опционально, для чистоты)
tb_forms['measure_type'] = tb_forms['measure_type'].replace({
    'Абс. числа': 'абс. число',
    'На 1000000 населения': 'на 100000 населения'
})

# 2. Доля женщин (только по абсолютным числам, но результат применим ко всем измерениям)
df_abs = tb_forms[tb_forms['measure_type'] == 'абс. число'].copy()

# Сводка: год, форма -> total (оба пола) и women
pivot_women = df_abs.pivot_table(
    index=['year', 'name'],
    columns='gender',
    values='value',
    aggfunc='first'
).reset_index()
pivot_women.columns.name = None
pivot_women.rename(columns={'оба пола': 'total', 'женщины': 'women'}, inplace=True)
pivot_women['women_percent'] = (pivot_women['women'] / pivot_women['total'] * 100).round(1)

# Присоединяем women_percent к исходному датасету (по year и name)
tb_forms = tb_forms.merge(
    pivot_women[['year', 'name', 'women_percent']],
    on=['year', 'name'],
    how='left'
)

# 3. Доля детей 0-14 лет (общая, не зависит от формы)
# Берём абсолютные числа для "все формы туберкулеза у детей 0-14" и "все формы туберкулеза"
child_total = tb_forms[
    (tb_forms['name'] == 'все формы туберкулеза у детей в возрасте 0-14 лет включительно') &
    (tb_forms['measure_type'] == 'абс. число') &
    (tb_forms['gender'] == 'оба пола')
][['year', 'value']].rename(columns={'value': 'children_abs'})

all_forms_total = tb_forms[
    (tb_forms['name'] == 'все формы туберкулеза') &
    (tb_forms['measure_type'] == 'абс. число') &
    (tb_forms['gender'] == 'оба пола')
][['year', 'value']].rename(columns={'value': 'total_abs'})

# Объединяем и вычисляем процент
child_share = pd.merge(child_total, all_forms_total, on='year', how='inner')
child_share['children_percent'] = (child_share['children_abs'] / child_share['total_abs'] * 100).round(1)

# Присоединяем children_percent ко всем строкам исходного датасета (по year)
tb_forms = tb_forms.merge(
    child_share[['year', 'children_percent']],
    on='year',
    how='left'
)

# 4. Сохраняем результат
tb_forms.to_csv('tb_forms_with_shares.csv', index=False, encoding='utf-8-sig')

print("✅ Файл tb_forms_with_shares.csv сохранён.")
print(f"Колонки: {list(tb_forms.columns)}")
print(f"Пример значений women_percent и children_percent:")
print(tb_forms[['year', 'name', 'women_percent', 'children_percent']].drop_duplicates().head(10))

✅ Файл tb_forms_with_shares.csv сохранён.
Колонки: ['year', 'name', 'full_name', 'gender', 'measure_type', 'value', 'kids_adult', 'women_percent', 'children_percent']
Пример значений women_percent и children_percent:
    year                                               name  women_percent  \
0   2015                              все формы туберкулеза           31.8   
1   2015                         туберкулез органов дыхания           31.2   
2   2015                             внелегочный туберкулез           47.6   
3   2015                 туберкулез мозговых оболочек и цнс           38.8   
4   2015                       туберкулез костей и суставов           35.1   
5   2015                     туберкулез мочеполовых органов           59.1   
6   2015               туберкулез периферических лимфоузлов           53.8   
7   2015  все формы туберкулеза у детей в возрасте 0-14 ...           51.1   
16  2016                              все формы туберкулеза           32.2   
17 

### Создание датасета с индексом хронизации

In [12]:

inc_name = 'Впервые установленный диагноз'
prev_name = 'Под диспансерным наблюдением на конец года'

df = tb_regions[tb_regions['name'].isin([inc_name, prev_name])].copy()

df_abs = df[df['measure_type'] == 'абс. число']
df_per = df[df['measure_type'] == 'на 100000 населения']

pivot_abs = df_abs.pivot_table(
    index=['region', 'year'],
    columns='name',
    values='value',
    aggfunc='first'
).reset_index()
pivot_abs.columns.name = None
pivot_abs.rename(columns={inc_name: 'incidence_abs', prev_name: 'prevalence_abs'}, inplace=True)

pivot_per = df_per.pivot_table(
    index=['region', 'year'],
    columns='name',
    values='value',
    aggfunc='first'
).reset_index()
pivot_per.columns.name = None
pivot_per.rename(columns={inc_name: 'incidence_per100k', prev_name: 'prevalence_per100k'}, inplace=True)

df_wide = pd.merge(pivot_abs, pivot_per, on=['region', 'year'], how='outer')

# Население
df_wide['population'] = (df_wide['incidence_abs'] * 100000) / df_wide['incidence_per100k'].replace(0, np.nan)
df_wide['population'] = df_wide['population'].round(0)

# Индекс хронизации
df_wide['chronic_index'] = df_wide['prevalence_per100k'] / df_wide['incidence_per100k'].replace(0, np.nan)
df_wide['chronic_index'] = df_wide['chronic_index'].round(2)

records = []
for _, row in df_wide.iterrows():
    r = row['region']
    y = row['year']
    pop = row['population'] if pd.notna(row['population']) else None

    # Абсолютные значения
    if pd.notna(row.get('incidence_abs')):
        records.append({'region': r, 'year': y, 'measure_type': 'абс. число', 'metric': 'incidence', 'value': row['incidence_abs'], 'population': pop})
    if pd.notna(row.get('prevalence_abs')):
        records.append({'region': r, 'year': y, 'measure_type': 'абс. число', 'metric': 'prevalence', 'value': row['prevalence_abs'], 'population': pop})

    # Относительные значения
    if pd.notna(row.get('incidence_per100k')):
        records.append({'region': r, 'year': y, 'measure_type': 'на 100000 населения', 'metric': 'incidence', 'value': row['incidence_per100k'], 'population': pop})
    if pd.notna(row.get('prevalence_per100k')):
        records.append({'region': r, 'year': y, 'measure_type': 'на 100000 населения', 'metric': 'prevalence', 'value': row['prevalence_per100k'], 'population': pop})

    # Индекс хронизации – дублируем для обоих measure_type
    if pd.notna(row.get('chronic_index')):
        # Для относительных
        records.append({'region': r, 'year': y, 'measure_type': 'на 100000 населения', 'metric': 'chronic_index', 'value': row['chronic_index'], 'population': pop})
        # Для абсолютных (значение то же, population оставляем pop, но можно None)
        records.append({'region': r, 'year': y, 'measure_type': 'абс. число', 'metric': 'chronic_index', 'value': row['chronic_index'], 'population': pop})

df_long = pd.DataFrame(records)
df_long = df_long.sort_values(['region', 'year', 'measure_type', 'metric'])

# Сохранение
filename = 'tb_regions_index.csv'
df_long.to_csv(filename, index=False, encoding='utf-8-sig')
csv_data = df_long.to_csv(index=False, encoding='utf-8-sig')
b64 = base64.b64encode(csv_data.encode()).decode()
display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}">📥 Скачать {filename}</a>'))
print(f"✅ {filename} сохранён, строк: {df_long.shape[0]}, колонок: {df_long.shape[1]}")

✅ tb_regions_index.csv сохранён, строк: 5076, колонок: 6


In [13]:
moscow = tb_regions[(tb_regions['region']=='Москва') & 
                    (tb_regions['year']==2020) & 
                    (tb_regions['measure_type']=='на 100000 населения')]
print(moscow[['name', 'value']])

Empty DataFrame
Columns: [name, value]
Index: []


### Полигоны регионов

👉 https://storage.yandexcloud.net/doc-files/Regions.csv

Это официальный файл от Яндекса, предназначенный для использования в DataLens. Он содержит правильные названия регионов и геометрию в формате, который DataLens понимает «из коробки» (поле Полигон с типом «Геополигон»).

#### Загрузка

In [14]:
# Загружает все датасеты
df_indicators_sub = pd.read_csv('df_indicators_sub.csv')
irkutsk_population_2016_2024 = pd.read_csv('irkutsk_population_2016_2024.csv')

print("Все файлы загружены!")
print("\n" + "="*80)
print("РАЗМЕР ЗАГРУЖЕННЫХ ДАТАСЕТОВ")
print("="*80)

# Список всех загруженных переменных (исключая служебные)
dataset_names = ['df_indicators_sub', 'irkutsk_population_2016_2024']

for name in dataset_names:
    df = globals()[name]
    print(f"{name:<20} → {len(df):>6} строк, {len(df.columns):>3} колонок")

print("="*80)

Все файлы загружены!

РАЗМЕР ЗАГРУЖЕННЫХ ДАТАСЕТОВ
df_indicators_sub    →   1406 строк,  14 колонок
irkutsk_population_2016_2024 →     27 строк,  17 колонок


In [15]:
# ======================================================================
# ЗАГРУЗКА ФАЙЛА Regions_polygons.csv (С УЧЁТОМ ОСОБЕННОСТЕЙ ФОРМАТА)
# ======================================================================
# 1. Определяем кодировку
with open('Regions_polygons.csv', 'rb') as f:
    raw = f.read(50000)
    encoding = chardet.detect(raw)['encoding']
    print(f"Кодировка: {encoding}")

# 2. Загружаем с правильными параметрами
polygons = pd.read_csv('Regions_polygons.csv', 
                       encoding=encoding,
                       sep=';',           # разделитель - точка с запятой
                       engine='python',
                       header=0,
                       on_bad_lines='skip')

print(f"Загружено {len(polygons)} строк")
print(f"Колонки: {list(polygons.columns)}")
display(polygons.head(3))

# 3. Переименовываем колонки (как в прошлый раз)
polygons = polygons.rename(columns={
    'Регион ДТП': 'region',
    'Полигон': 'polygon'
})

print(f"\nПосле переименования колонки: {list(polygons.columns)}")
display(polygons.head(2))

Кодировка: windows-1251
Загружено 85 строк
Колонки: ['Регион ДТП', 'Полигон']


,Регион ДТП,Полигон
0,Архангельская область,"[[[65.051193, 35.314801], [65.127703, 35.29793..."
1,Алтайский край,"[[[53.290641, 77.907506], [53.367469, 77.88941..."
2,Амурская область,"[[[56.865828, 119.818979], [56.903958, 119.884..."



После переименования колонки: ['region', 'polygon']


,region,polygon
0,Архангельская область,"[[[65.051193, 35.314801], [65.127703, 35.29793..."
1,Алтайский край,"[[[53.290641, 77.907506], [53.367469, 77.88941..."


#### Нормализация для объединения

In [16]:
# ======================================================================
# СРАВНЕНИЕ УНИКАЛЬНЫХ РЕГИОНОВ В tb_regions И polygons
# ======================================================================

# Уникальные регионы из tb_regions
tb_regions_unique = sorted(tb_regions['region'].unique())
print("="*80)
print(f"РЕГИОНЫ В ТАБЛИЦЕ СТАТИСТИКИ (tb_regions): {len(tb_regions_unique)}")
print("="*80)
for i, r in enumerate(tb_regions_unique, 1):
    print(f"{i:3d}. {r}")

# Уникальные регионы из polygons (после загрузки и переименования)
polygons_regions_unique = sorted(polygons['region'].unique())
print("\n" + "="*80)
print(f"РЕГИОНЫ В ТАБЛИЦЕ ПОЛИГОНОВ: {len(polygons_regions_unique)}")
print("="*80)
for i, r in enumerate(polygons_regions_unique, 1):
    print(f"{i:3d}. {r}")

# Сравнение
missing_in_polygons = set(tb_regions_unique) - set(polygons_regions_unique)
print("\n" + "="*80)
print(f"РЕГИОНЫ, КОТОРЫЕ ЕСТЬ В СТАТИСТИКЕ, НО ОТСУТСТВУЮТ В ПОЛИГОНАХ: {len(missing_in_polygons)}")
print("="*80)
for r in sorted(missing_in_polygons):
    print(f"   • {r}")

missing_in_stats = set(polygons_regions_unique) - set(tb_regions_unique)
print("\n" + "="*80)
print(f"РЕГИОНЫ, КОТОРЫЕ ЕСТЬ В ПОЛИГОНАХ, НО ОТСУТСТВУЮТ В СТАТИСТИКЕ: {len(missing_in_stats)}")
print("="*80)
for r in sorted(missing_in_stats):
    print(f"   • {r}")

РЕГИОНЫ В ТАБЛИЦЕ СТАТИСТИКИ (tb_regions): 94
  1. Алтайский край
  2. Амурская область
  3. Архангельская область без автономного округа
  4. Астраханская область
  5. Белгородская область
  6. Брянская область
  7. Владимирская область
  8. Волгоградская область
  9. Вологодская область
 10. Воронежская область
 11. Дальневосточный федеральный округ
 12. Еврейская автономная область
 13. Забайкальский край
 14. Ивановская область
 15. Иркутская область
 16. Кабардино-Балкарская Республика
 17. Калининградская область
 18. Калужская область
 19. Камчатский край
 20. Карачаево-Черкесская Республика
 21. Кемеровская область
 22. Кировская область
 23. Костромская область
 24. Краснодарский край
 25. Красноярский край
 26. Курганская область
 27. Курская область
 28. Ленинградская область
 29. Липецкая область
 30. Магаданская область
 31. Московская область
 32. Мурманская область
 33. Ненецкий автономный округ
 34. Нижегородская область
 35. Новгородская область
 36. Новосибирская обла

In [17]:
# ======================================================================
# ПРИВЕДЕНИЕ tb_regions К ФОРМАТУ polygons (БЕЗ УДАЛЕНИЯ)
# ======================================================================

# Копируем исходные данные
tb = tb_regions.copy()
poly = polygons.copy()

# Приводим названия к формату polygons (без удаления округов и РФ)
replacements = {
    'город Москва': 'Москва',
    'город Санкт-Петербург': 'Санкт-Петербург',
    'город Севастополь': 'Севастополь',
    'Архангельская область без автономного округа': 'Архангельская область',
    'Тюменская область без автономного округа': 'Тюменская область',
    'Ханты-Мансийский АО': 'Ханты-Мансийский автономный округ — Югра',
    'Ямало-Ненецкий АО': 'Ямало-Ненецкий автономный округ',
    'Республика Северная Осетия - Алания': 'Республика Северная Осетия — Алания',
    'Чувашская Республика': 'Чувашская Республика — Чувашия',
}
tb['region'] = tb['region'].replace(replacements)

# Обновляем исходные датасеты
tb_regions = tb
polygons = poly

print("✅ Названия регионов в tb_regions приведены к формату polygons.")
print("Федеральные округа и РФ остались в датасете (не удалены).")
print(f"\nУникальных регионов в tb_regions: {tb_regions['region'].nunique()}")
print(f"Уникальных регионов в polygons: {polygons['region'].nunique()}")

# Теперь датасеты готовы для JOIN по полю 'region'
# В DataLens при построении карты можно будет отфильтровать округа и РФ

✅ Названия регионов в tb_regions приведены к формату polygons.
Федеральные округа и РФ остались в датасете (не удалены).

Уникальных регионов в tb_regions: 94
Уникальных регионов в polygons: 84


In [18]:
print("Уникальные регионы в tb_regions:")
display(tb_regions['region'].unique())

print("\nУникальные регионы в polygons:")
display(polygons['region'].unique())

Уникальные регионы в tb_regions:


array(['Российская Федерация', 'Центральный федеральный округ',
       'Белгородская область', 'Брянская область', 'Владимирская область',
       'Воронежская область', 'Ивановская область', 'Калужская область',
       'Костромская область', 'Курская область', 'Липецкая область',
       'Московская область', 'Орловская область', 'Рязанская область',
       'Смоленская область', 'Тамбовская область', 'Тверская область',
       'Тульская область', 'Ярославская область', 'Москва',
       'Северо-Западный федеральный округ', 'Республика Карелия',
       'Республика Коми', 'Архангельская область',
       'Ненецкий автономный округ', 'Вологодская область',
       'Калининградская область', 'Ленинградская область',
       'Мурманская область', 'Новгородская область', 'Псковская область',
       'Санкт-Петербург', 'Южный федеральный округ', 'Республика Адыгея',
       'Республика Калмыкия', 'Республика Крым', 'Краснодарский край',
       'Астраханская область', 'Волгоградская область',
       


Уникальные регионы в polygons:


array(['Архангельская область', 'Алтайский край', 'Амурская область',
       'Астраханская область', 'Белгородская область', 'Брянская область',
       'Владимирская область', 'Волгоградская область',
       'Вологодская область', 'Воронежская область', 'Москва',
       'Еврейская автономная область', 'Забайкальский край',
       'Ивановская область', 'Иркутская область',
       'Кабардино-Балкарская Республика', 'Калининградская область',
       'Калужская область', 'Камчатский край',
       'Карачаево-Черкесская Республика', 'Кемеровская область',
       'Кировская область', 'Костромская область', 'Краснодарский край',
       'Красноярский край', 'Курганская область', 'Курская область',
       'Ленинградская область', 'Липецкая область', 'Магаданская область',
       'Московская область', 'Мурманская область',
       'Ямало-Ненецкий автономный округ', 'Нижегородская область',
       'Новгородская область', 'Новосибирская область', 'Омская область',
       'Оренбургская область', 'Орл

В polygons есть 2 строки 'Ямало-Ненецкий автономный округ', что по всей видимости (как видно по геокарте в Datalens) свидетельствует о том, что одна из них 'Ненецкий автономный округ', который находится западнее. Проанализировав минимальную долготу мы определили какую строку переименовать.

In [19]:
# Переименовываем строку с индексом 32 в 'Ненецкий автономный округ'
polygons.loc[32, 'region'] = 'Ненецкий автономный округ'

In [20]:

# Находим уникальные значения в каждом
vals_tb = set(tb_regions['region'].dropna())
vals_poly = set(polygons['region'].dropna())

# Выводим только расходящиеся значения для каждого датасета
print("В tb_regions, но НЕ в polygons:")
print(vals_tb - vals_poly)

print("\nВ polygons, но НЕ в tb_regions:")
print(vals_poly - vals_tb)

В tb_regions, но НЕ в polygons:
{'Южный федеральный округ', 'Дальневосточный федеральный округ', 'Российская Федерация', 'Уральский федеральный округ', 'Приволжский федеральный округ', 'Северо-Кавказский федеральный округ', 'Центральный федеральный округ', 'Сибирский федеральный округ', 'Северо-Западный федеральный округ'}

В polygons, но НЕ в tb_regions:
set()


### Добавляем value_prev. Колонку со значения ми предыдущего года

In [21]:
tb_russia.columns

Index(['year', 'name', 'measure_type', 'value'], dtype='object')

In [22]:
# Сортируем по name, measure_type, year
tb_russia = tb_russia.sort_values(['name', 'measure_type', 'year'])

# Создаём колонку с предыдущим значением (по группам name + measure_type)
tb_russia['val_prev'] = tb_russia.groupby(['name', 'measure_type'])['value'].shift(1)

# Для первого года в группе (нет предыдущего) ставим 0
tb_russia['val_prev'] = tb_russia['val_prev'].fillna(0)

print("✅ Колонка val_prev добавлена")


✅ Колонка val_prev добавлена


In [23]:
# Правильная сортировка: сначала регион, name, measure_type, потом год
tb_regions = tb_regions.sort_values(['region', 'name', 'measure_type', 'year'])

# Теперь shift(1) будет корректно брать предыдущий год
tb_regions['val_prev'] = tb_regions.groupby(['region', 'name', 'measure_type'])['value'].shift(1)

# Заполняем 0 для первого года
tb_regions['val_prev'] = tb_regions['val_prev'].fillna(0)


In [24]:
# Сортируем по year, name, measure_type, gender
tb_forms = tb_forms.sort_values(['year', 'name', 'measure_type', 'gender'])

# Создаём колонку с предыдущим значением
tb_forms['val_prev'] = tb_forms.groupby(['name', 'measure_type', 'gender'])['value'].shift(1)

# Проверка: если разница между текущим и предыдущим годом не равна 1, то val_prev = 0
tb_forms['year_prev'] = tb_forms.groupby(['name', 'measure_type', 'gender'])['year'].shift(1)
tb_forms['year_diff'] = tb_forms['year'] - tb_forms['year_prev']

# Обнуляем val_prev, если разница в годах не 1
tb_forms.loc[tb_forms['year_diff'] != 1, 'val_prev'] = 0

# Для первого значения в группе ставим 0
tb_forms['val_prev'] = tb_forms['val_prev'].fillna(0)

# Удаляем временные колонки
tb_forms = tb_forms.drop(columns=['year_prev', 'year_diff'])



### Cохраняем

In [25]:
# Список всех датасетов для сохранения
all_datasets = [
    ('tb_russia', tb_russia),
    ('tb_regions', tb_regions),
    ('tb_forms', tb_forms),
    ('polygons', polygons),
#    ('tb_regions_polygons', tb_regions_polygons),
    ('irkutsk_population_2016_2024', irkutsk_population_2016_2024),
]

# Функция для создания ссылки на скачивание
def create_download_link(df, filename):
    csv = df.to_csv(index=False, encoding='utf-8-sig')
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}">📥 Скачать {filename}</a>'))

print("="*80)
print("СОХРАНЕНИЕ ТАБЛИЦ В CSV И ССЫЛКИ ДЛЯ СКАЧИВАНИЯ")
print("="*80)

for name, df in all_datasets:
    if df is not None:
        filename = f'{name}.csv'
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        create_download_link(df, filename)
        print(f"✅ {filename} сохранён и доступен по ссылке выше")
        print(f"   Размер: {df.shape[0]} строк × {df.shape[1]} колонок\n")
    else:
        print(f"⚠️ {name} не определён\n")

print("="*80)
print("✅ Готово! Все файлы сохранены в текущую директорию и доступны для скачивания.")
print("="*80)

print("\n📊 Информация о переменных:")
for name, df in all_datasets:
    if df is not None:
        print(f"   {name}: {df.shape[0]} строк, {df.shape[1]} колонок")

СОХРАНЕНИЕ ТАБЛИЦ В CSV И ССЫЛКИ ДЛЯ СКАЧИВАНИЯ


✅ tb_russia.csv сохранён и доступен по ссылке выше
   Размер: 256 строк × 5 колонок



✅ tb_regions.csv сохранён и доступен по ссылке выше
   Размер: 3384 строк × 7 колонок



✅ tb_forms.csv сохранён и доступен по ссылке выше
   Размер: 540 строк × 10 колонок



✅ polygons.csv сохранён и доступен по ссылке выше
   Размер: 85 строк × 2 колонок



✅ irkutsk_population_2016_2024.csv сохранён и доступен по ссылке выше
   Размер: 27 строк × 17 колонок

✅ Готово! Все файлы сохранены в текущую директорию и доступны для скачивания.

📊 Информация о переменных:
   tb_russia: 256 строк, 5 колонок
   tb_regions: 3384 строк, 7 колонок
   tb_forms: 540 строк, 10 колонок
   polygons: 85 строк, 2 колонок
   irkutsk_population_2016_2024: 27 строк, 17 колонок
